In [22]:
import numpy as np
import os, sys
from CoolProp.HumidAirProp import HAPropsSI

In [4]:
V_room = 10.0 * 6.0 * 3.0       #[m^3]
p_atm = 101.3250                #[kPa]
R_air = 0.287                   #[kJ/(kg*K)]
T_air = 273.15 + 15             #[K]            
m_air_room = p_atm * V_room / (R_air * T_air)

cv_air = 0.718              #[kJ/(kg*K)]
cp_air = 1.005              #[kJ/(kg*K)]

In [11]:
m_dot = 1.8                     #[kg/s]
P_vent = 0.09                   #[kW]

Q_dot_vent_cool_min = -3 #[kW]

T_COOL_ON = 17            #[°C]
T_COOL_OFF = 13            #[°C]

In [ ]:
def calc_Q_dot_vent_cool(T_room: float, T_amb: float, m_dot: float):
    # Following calculate the heat removal when ventilation is on
    # This is base on the Energy Balance
    Q_dot_vent_cool = m_dot * cp_air * (T_amb - T_room) #[kW]
    return Q_dot_vent_cool


def if_vent_useful(Q_dot_vent_cool: float, Q_dot_server_est: float):
    if (Q_dot_vent_cool + Q_dot_server_est) < -1:
        return True
    else:
        return False

In [8]:
def server_power_estimater(T_room_curr: float, T_room_prev: float, T_amb: float, Q_dot_AC: float, mode: int, dt: float):
    '''
    mode = 0: nothing, 1: ventilation, 2: AC ON
    '''
    if mode == 0:
        Q_dot_server_est = m_air_room * cv_air * (T_room_curr - T_room_prev)/dt
        return Q_dot_server_est
    elif mode == 1:
        Q_dot_server_est = m_air_room * cv_air * (T_room_curr - T_room_prev)/dt + m_dot * cp_air * (T_room_prev - T_amb)
        return Q_dot_server_est
    elif mode == 2:
        Q_dot_server_est = m_air_room * cv_air * (T_room_curr - T_room_prev)/dt - Q_dot_AC
        return Q_dot_server_est

In [ ]:
def controller(T_room_curr: float, T_room_prev: float, T_amb: float, Q_dot_AC: float, mode: int, dt: float):
    # Estimate the server power first
    Q_dot_server_est = server_power_estimater(T_room_curr, T_room_prev, T_amb, Q_dot_AC, mode, dt)
    
    # calculate how much heat removal the ventilation can supply
    Q_dot_vent_cool = calc_Q_dot_vent_cool(T_room_curr, T_amb, m_dot)
    
    cool_tick = 0
    # Check room temperature is at which range
    if T_room_curr > 17:
        # Critical, need full power cooling
        # Check which one is better, AC or vent
        cool_tick = 3
    if T_room_curr > 16 and T_room_curr <= 17:
        # somewhat okay but better bring it down, AC or vent
        cool_tick = 2
    if T_room_curr > 15 and T_room_curr <= 16:
        # indicate small cooling needed, or do nothing
        cool_tick = 1
    if T_room_curr <= 15 and T_room_curr > 14:
        # below target, can continue run the current mode, AC or vent for a bit
        cool_tick = -1
    if T_room_curr <= 14 and T_room_curr > 13:
        # a little bit chill now, can stop cooling, if vent is on or AC when the temperature is rising, can let it run for a bit
        cool_tick = -2
    if T_room_curr <= 13:
        # Too cold, stop all cooling
        cool_tick = -3
    
    # Estimation of next step temperature by applying different cooling or do nothing
    
    # Do nothing
    T_next_OFF = T_room_curr + Q_dot_server_est * dt /(m_air_room * cv_air)
    # Vent
    T_next_VEN = T_room_curr + (T_amb - T_room_curr) * m_dot * cp_air * dt / (m_air_room * cv_air) + Q_dot_server_est * dt / (m_air_room * cv_air)
    # AC
    T_next_AC = T_room_curr + (Q_dot_server_est + Q_dot_AC) * dt / (m_air_room * cv_air) 
    
    # Check which one gets closest to the target temperature of 15C
    Delta_T_OFF = abs(T_next_OFF - 15)
    Delta_T_VEN = abs(T_next_VEN - 15)
    Delta_T_AC = abs(T_next_AC - 15)
    
    # Pack into an array with labels
    deltas = np.array([Delta_T_OFF, Delta_T_VEN, Delta_T_AC])
    labels = ['OFF', 'VEN', 'AC']
    
    # Find the index of the minimum value
    idx = np.argmin(deltas)
    
    # Get the minimum value and its label
    min_delta = deltas[idx]
    best_action = labels[idx]
    
    if best_action == "OFF":
        T_next = T_next_OFF
    if best_action == "VEN":
        T_next = T_next_VEN
    if best_action == "AC":
        T_next = T_next_AC
    
    # But we also have to look at the cool_tick, how critical it is to change state
    # This is to protect the AC on off too often, so if the temperature of next action is in the buffer zone, can leave the AC running
    # Check room temperature is at which range
    cool_tick_next = 0
    if T_next > 17:
        # Critical, need full power cooling
        # Check which one is better, AC or vent
        cool_tick_next = 3
    if T_next > 16 and T_next <= 17:
        # somewhat okay but better bring it down, AC or vent
        cool_tick_next = 2
    if T_next > 15 and T_next <= 16:
        # indicate small cooling needed, or do nothing
        cool_tick_next = 1
    if T_next <= 15 and T_next > 14:
        # below target, can continue run the current mode, AC or vent for a bit
        cool_tick_next = -1
    if T_next <= 14 and T_next > 13:
        # a little bit chill now, can stop cooling, if vent is on or AC when the temperature is rising, can let it run for a bit
        cool_tick_next = -2
    if T_next <= 13:
        # Too cold, stop all cooling
        cool_tick_next = -3
        
        
    return idx
    
    if cool_tick_next == 3:
        return idx
    if cool_tick_next == 2:
        if mode == 2:
            return 2

    

In [ ]:
dt = 150
i = int(24 * 3600 / dt)   # timesteps per day (576)
T_initial  = 15.0                # °C, initial room temperature
Rh_initial = 0.60                # initial relative humidity
Rh_Outdoor = 0.60                # outdoor relative humidity

AC_PERFORMANCE = {
    "Propane":  {30: (3.99, 4.0345), 40: (3.99, 7.17245), 50: (3.99, 11.20695)},
    "R1234yf":  {30: (3.83, 2.84624), 40: (3.83, 5.05999), 50: (3.83, 7.90623)},
    "DME":      {30: (4.07, 2.82896), 40: (4.07, 5.02926), 50: (4.07, 7.85822)},
}

In [ ]:
def interpolate(data: np.ndarray) -> np.ndarray:
    raw       = np.linspace(0, 1, len(data))
    processed = np.linspace(0, 1, i)
    return np.interp(processed, raw, data)

def w_from_rh(T_C: float, RH: float) -> float:
    return HAPropsSI("W", "T", T_C + 273.15, "P", p_atm, "R", RH)

In [ ]:
# Set up simulation
def sim(T_amb: np.ndarray, Q_server: np.ndarray, refrigerant: str,
        cylinder_size: float, T_init: float = T_initial) -> dict:
    
    COP_inner, Q_dot_AC = AC_PERFORMANCE[refrigerant][cylinder_size]
    
    T_amb    = interpolate(T_amb)
    Q_server = interpolate(Q_server)
    
    
    T_room = np.empty(i);  T_room[0] = T_init
    w_room = np.empty(i);  w_room[0] = w_from_rh(T_init, Rh_initial)
    mode   = np.empty(i, dtype=int)
    mode[0]= 0
    Q_cool = np.zeros(i)
    W_comp = np.zeros(i)
    W_vent = np.zeros(i)
    ac_plr = np.zeros(i)
    
    for k in range(i - 1):
        mode[k+1] = controller(T_room[k+1],T_room[k],T_amb[k+1],Q_dot_AC,mode[k],dt)
        
        if mode[k+1] == 0:
            T_room[k+2] = T_room[k+1] + Q_server[k+1] * dt /(m_air_room * cv_air)
            
        if mode[k+1] == 1:
            T_room[k+2] = T_room[k+1] + (T_amb - T_room[k+1]) * m_dot * cp_air * dt / (m_air_room * cv_air) + Q_server[k+1] * dt / (m_air_room * cv_air)
            
        if mode[k+1] == 2:
            T_room[k+2] = T_room[k+1] + (Q_server[k+1] + Q_dot_AC) * dt / (m_air_room * cv_air) 